In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis
import warnings

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

# 다이캐스팅 품질 데이터 전처리

## 1. 데이터 로드 및 기본 확인

In [ ]:
# 데이터셋 로드
print("="*60)
df_full = pd.read_csv('../../data/DieCasting_Quality_Raw_Data.csv', header=[0, 1])
print("데이터셋 로드 완료!")

# 컬럼 이름에 공백 있는 컬럼 공백 제거
df_full.columns = pd.MultiIndex.from_tuples(
    [(group.strip(), col.strip()) for group, col in df_full.columns]
)
print("컬럼 이름 공백 제거 완료!")
print("="*60)

In [ ]:
# 데이터셋 정보 확인
print(f"Shape: {df_full.shape}")
print("\n[Data Info]")
df_full.info()

In [ ]:
# 원본 데이터 기초 통계 확인
print("\n" + "="*60)
print("다이캐스팅 데이터셋 기초 통계")
print("="*60)
display(df_full.describe(include='all').T)

In [ ]:
# 데이터셋 샘플 확인
print("\n" + "="*60)
print("DieCasting 샘플 데이터")
print("="*60)
display(df_full.head())

## 2. 데이터 전처리

### 2-1. 중복 데이터 확인

In [ ]:
print(f"전체 중복 데이터 개수: {df_full.duplicated().sum()}")
id_dup = df_full.duplicated(subset=[('Process', 'id')]).sum()
print(f"(process, id) 기준 중복 데이터 개수: {id_dup}")

### 2-2. 결측치 확인 및 처리

In [ ]:
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': df_full.isnull().sum(),
    '결측비율(%)': (df_full.isnull().sum() / len(df_full) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")

In [ ]:
# Sensor 컬럼 중 결측치가 있는 컬럼들 추출
sensor_cols_with_na = [col for col in df_full['Sensor'].columns 
                       if df_full[('Sensor', col)].isna().sum() > 0]

# 각 컬럼의 중앙값으로 결측치 채우기
for col in sensor_cols_with_na:
    median_val = df_full[('Sensor', col)].median()
    df_full[('Sensor', col)] = df_full[('Sensor', col)].fillna(median_val)
    print(f"'{col}' -> 중앙값 {median_val}로 대채")

print("="*60)
print("결측치 중앙값 대체 완료!")
print("="*60)

### 2-3. Defects(불량품) 컬럼 이상치 확인 및 처리

In [ ]:
# 불량품 관련 컬럼(Defects) 중 값이 '1' 초과인 컬럼과 갯수 출력
defect_cols = df_full['Defects'].columns

total_sum = 0
over_one_cnt = {}
for col in defect_cols:
    over_one = df_full[('Defects', col)][df_full[('Defects', col)] > 1]
    if len(over_one) > 0:
        over_one_cnt[col] = over_one.value_counts().sort_index()

if over_one_cnt:
    print("="*40)
    print("값이 1 초과인 Defects 컬럼 목록:")
    print("="*40)
    for col, val_counts in over_one_cnt.items():
        total = val_counts.sum()
        total_sum += total
        values_str = ", ".join([f"{v}→{c}개" for v, c in val_counts.items()])
        print(f"  {col}: 총 {total}개  ({values_str})")
else:
    print("값이 1 초과인 Defects 컬럼이 없습니다.")


total_defect_values = len(df_full) * len(defect_cols)  # 전체 Defects 데이터 수 (행 × 컬럼)
ratio = total_sum / total_defect_values * 100

print("")
print("="*40)
print(f"Defects 컬럼 총 이상치 개수: {total_sum}개")
print(f"전체 Defects 데이터 수: {total_defect_values}개 ({len(df_full)}행 × {len(defect_cols)}컬럼)")
print(f"Defects 컬럼 이상치 비율: {ratio:.4f}%")
print("="*40)

In [ ]:
# Defects 관련 컬럼 값 중 1을 초과하는 값은 1(불량)으로 대체
replaced = False
for col in defect_cols:
    mask = df_full[('Defects', col)] > 1
    if mask.any():
        df_full.loc[mask, ('Defects', col)] = 1
        print(f"{col}: {mask.sum()}개 값을 1로 대체")
        replaced = True

print("="*60)
if not replaced:
    print("대체 할 이상치 값이 없습니다.")
else:
    print("값 대체 완료!")
print("="*60)

## 3. 이상치 분석

In [ ]:
# product_type 별 데이터 분리
product_type_1_df = df_full[df_full[('Process', 'Product_Type')] == 1]
product_type_2_df = df_full[df_full[('Process', 'Product_Type')] == 2]

print(f"Product_Type 1: {len(product_type_1_df)}행")
print(f"Product_Type 2: {len(product_type_2_df)}행")

### 3-1. 불량 유형별 불량률 확인

In [ ]:
# Product_Type별 불량 컬럼 불량률 확인
type_dfs   = [product_type_1_df, product_type_2_df]
type_names = ['Type 1', 'Type 2']

fig, axes = plt.subplots(2, 1, figsize=(14, 12))

for ax, type_df, type_name in zip(axes, type_dfs, type_names):
    defect_cols = type_df['Defects'].columns
    defect_rates = {col: type_df[('Defects', col)].mean() * 100 for col in defect_cols if type_df[('Defects', col)].any()}
    defect_rates = pd.Series(defect_rates).sort_values(ascending=False)

    sns.barplot(x=defect_rates.index, y=defect_rates.values, palette='viridis', ax=ax)
    ax.set_title(f'[{type_name}] 불량 컬럼별 불량률 (%)', fontsize=14, fontweight='bold')
    ax.set_xlabel('불량 컬럼', fontsize=11)
    ax.set_ylabel('불량률 (%)', fontsize=11)
    ax.set_xticks(range(len(defect_rates)))
    ax.set_xticklabels(defect_rates.index, rotation=45, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 전체 정상/불량률 확인
# 레이아웃: 행=Type, 열=[파이차트 | 바차트]
type_dfs   = [product_type_1_df, product_type_2_df]
type_names = ['Type 1', 'Type 2']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (type_df, type_name) in enumerate(zip(type_dfs, type_names)):
    is_defect = type_df['Defects'].any(axis=1)
    defect_count = is_defect.sum()
    normal_count = (~is_defect).sum()
    total = len(type_df)

    # 파이차트 (col 0)
    axes[i, 0].pie(
        [normal_count, defect_count],
        labels=['정상', '불량'],
        autopct='%1.1f%%',
        startangle=90,
        counterclock=False,
        colors=['steelblue', 'tomato'],
    )
    axes[i, 0].set_title(f'[{type_name}] 정상/불량 비율\n(전체 {total}행)', fontsize=12, fontweight='bold')

    # 바차트 (col 1)
    axes[i, 1].bar(['정상', '불량'], [normal_count, defect_count], color=['steelblue', 'tomato'], alpha=0.8)
    axes[i, 1].set_title(f'[{type_name}] 정상/불량 건수', fontsize=12, fontweight='bold')
    axes[i, 1].set_ylabel('건수')
    for j, v in enumerate([normal_count, defect_count]):
        axes[i, 1].text(j, v + total * 0.005, f'{v}\n({v/total*100:.1f}%)', ha='center', fontsize=10)
    axes[i, 1].grid(axis='y', alpha=0.3)

plt.suptitle('Product_Type별 정상/불량률', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3-2. Process(공정), Sensor(센서) 데이터 이상치 분석

In [ ]:
# Process(공정), Seonsor(센서) 데이터 이상치 탐색 컬럼 선택
target_cols = [
    ('Process', 'Velocity_1'), ('Process', 'Velocity_2'), ('Process', 'Velocity_3'),
    ('Process', 'High_Velocity'), ('Process', 'Cylinder_Pressure'), ('Process', 'Rapid_Rise_Time'),
    ('Process', 'Biscuit_Thickness'), ('Process', 'Clamping_Force'), ('Process', 'Cycle_Time'),
    ('Process', 'Pressure_Rise_Time'), ('Process', 'Casting_Pressure'), ('Process', 'Spray_Time'),
    ('Sensor', 'Melting_Furnace_Temp'), ('Sensor', 'Air_Pressure'),
    ('Sensor', 'Coolant_Temp'), ('Sensor', 'Coolant_Pressure'),
    ('Sensor', 'Factory_Temp'), ('Sensor', 'Factory_Humidity'),
]

# Process(공정), Seonsor(센서) 데이터 그래프 표시 정보
target_col_names = [
    '용탕 투입 속도 구간1', '용탕 투입 속도 구간2', '용탕 투입 속도 구간3',
    '고속 사출 속도', '사출 실린더 압력', '실린더 압력 급상승 시간',
    '주조 후 남은 비스킷 두께', '금형을 닫아주는 힘(형체력)', '1회 작업당 총 소요 시간',
    '최종 주조 압력까지 도달하는 시간', '용탕을 압착하는 최종 주조 압력', '금형 이형체 분무 전체 시간',
    '용해로 내부 용탕 온도', '작업용 에어 압력 측정값',
    '냉각수 온도 측정값', '냉각수 순환 압력',
    '공장 내부 온도 측정값', '공장 내부 습도 측정값',
]

# 기초통계
print("="*40)
print("Process/Sensor 데이터 기초통계 확인")
print("="*40)

df_full[target_cols].describe()

In [ ]:
# Product_Type별 왜도/첨도 확인
type_dfs   = [product_type_1_df, product_type_2_df]
type_names = ['Type 1', 'Type 2']

for type_df, type_name in zip(type_dfs, type_names):
    print("="*60)
    print(f"[{type_name}] 왜도/첨도 확인 (총 {len(type_df)}행)")
    print("="*60)

    for col, name in zip(target_cols, target_col_names):
        data = type_df[col]
        print(f"  {name} ({col[1]})")
        print(f"  범위: {data.min():.4f} ~ {data.max():.4f}")
        print(f"  왜도: {skew(data):.3f}  첨도: {kurtosis(data):.3f}")
        print()

In [ ]:
# Product_Type별 박스플롯 & 히스토그램 & Q-Q Plot
# 컬럼 구성: [박스플롯 | 히스토그램 | Q-Q Type1 | Q-Q Type2]
fig, axes = plt.subplots(len(target_cols), 4, figsize=(24, len(target_cols) * 3))

type_dfs   = [product_type_1_df, product_type_2_df]
type_names = ['Type 1', 'Type 2']
colors     = ['steelblue', 'darkorange']

for i, (col, name) in enumerate(zip(target_cols, target_col_names)):
    data_type1 = product_type_1_df[col]
    data_type2 = product_type_2_df[col]

    # 박스플롯 (col 0)
    bp = axes[i, 0].boxplot(
        [data_type1, data_type2],
        labels=['Type 1', 'Type 2'],
        patch_artist=True
    )
    for patch, color in zip(bp['boxes'], ['steelblue', 'darkorange']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[i, 0].set_title(f'{name}{col}')
    axes[i, 0].set_ylabel('값')
    axes[i, 0].grid(True, alpha=0.3)

    # 히스토그램 (col 1)
    axes[i, 1].hist(data_type1, bins='auto', alpha=0.5, color='steelblue', edgecolor='black', label='Type 1')
    axes[i, 1].hist(data_type2, bins='auto', alpha=0.5, color='darkorange', edgecolor='black', label='Type 2')
    axes[i, 1].set_title(f'{name}{col}')
    axes[i, 1].set_ylabel('빈도')
    axes[i, 1].legend(fontsize=9)
    axes[i, 1].grid(True, alpha=0.3)

    # Q-Q Plot (col 2: Type1, col 3: Type2)
    for qq_col, (type_df, type_name, color) in enumerate(zip(type_dfs, type_names, colors)):
        data = type_df[col].dropna()
        res = stats.probplot(data, dist='norm')

        theoretical = res[1]
        axes[i, 2 + qq_col].plot(
            [res[0][0].min(), res[0][0].max()],
            [theoretical[1] + theoretical[0] * res[0][0].min(),
             theoretical[1] + theoretical[0] * res[0][0].max()],
            color='red', linewidth=1.5
        )
        axes[i, 2 + qq_col].scatter(res[0][0], res[0][1], color=color, alpha=0.4, s=5)
        axes[i, 2 + qq_col].set_title(f'[{type_name}] {name}{col}\n(왜도: {skew(data):.3f})')
        axes[i, 2 + qq_col].set_xlabel('이론적 분위수')
        axes[i, 2 + qq_col].set_ylabel('실제 분위수')
        axes[i, 2 + qq_col].grid(True, alpha=0.3)

plt.suptitle('Product_Type별 공정/센서 변수 분포', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Product_Type별 IQR 기반 이상치 탐색
type_dfs   = [product_type_1_df, product_type_2_df]
type_names = ['Type 1', 'Type 2']

for type_df, type_name in zip(type_dfs, type_names):
    total_outlier_count = 0
    print(f"{'='*60}")
    print(f"[{type_name}] IQR 기반 이상치 탐색 (총 {len(type_df)}행)")
    print(f"{'='*60}")

    for col, name in zip(target_cols, target_col_names):
        data = type_df[col]
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outlier_count = ((data < lower_bound) | (data > upper_bound)).sum()
        total_outlier_count += outlier_count

        print(f"  {col}")
        print(f"  정상 범위: {lower_bound:.4f} ~ {upper_bound:.4f}")
        print(f"  이상치 개수: {outlier_count}개 ({outlier_count / len(type_df) * 100:.2f}%)\n")

    print(f"  총 이상치 합계: {total_outlier_count}개\n")

In [ ]:
type_dfs   = [product_type_1_df, product_type_2_df]
type_names = ['Type 1', 'Type 2']

print("="*60)
print("IQR 상한 값 보다 높은 값:")
print("="*60)
for type_df, type_name in zip(type_dfs, type_names):
    cycle_time_high = type_df[('Process', 'Cycle_Time')] > 100
    values = type_df.loc[cycle_time_high, ('Process', 'Cycle_Time')]
    print(f"[{type_name}] Cycle_Time > 100 인 값:")
    print(values.values)
    print()

print("="*60)
print("IQR 하한 값 보다 낮은 값:")
print("="*60)
for type_df, type_name in zip(type_dfs, type_names):
    cycle_time_high = type_df[('Process', 'Cycle_Time')] < 33
    values = type_df.loc[cycle_time_high, ('Process', 'Cycle_Time')]
    print(f"[{type_name}] Cycle_Time < 33 인 값:")
    print(values.values)
    print()

In [ ]:
# Product_Type별 Cycle_Time > 100인 행 불량 비율 비교
type_dfs   = [product_type_1_df, product_type_2_df]
type_names = ['Type 1', 'Type 2']

for type_df, type_name in zip(type_dfs, type_names):
    cycle_time_high = type_df[('Process', 'Cycle_Time')] > 100
    df_filtered = type_df[cycle_time_high]

    print(f"{'='*65}")
    print(f"[{type_name}] 전체: {len(type_df)}행 / Cycle_Time > 100: {len(df_filtered)}행 ({len(df_filtered)/len(type_df)*100:.2f}%)")
    print(f"{'='*65}")
    print(f"  {'컬럼':<20} {'전체 불량 비율':>15} {'Cycle_Time>100 불량 비율':>25}")
    print(f"  {'-'*60}")

    for col in df_full['Defects'].columns:
        total_ratio   = (type_df[('Defects', col)] == 1).sum() / len(type_df) * 100
        filtered_ratio = (df_filtered[('Defects', col)] == 1).sum() / len(df_filtered) * 100 if len(df_filtered) > 0 else 0
        if filtered_ratio > 0 or total_ratio > 0:
            print(f"  {col:<20} {total_ratio:>13.2f}%  {filtered_ratio:>23.2f}%")
    print()